In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a single Llama-style transformer decoder block. Given an input tensor $x$ of shape
  <code>(seq_len, 512)</code>, a packed weight buffer, and precomputed RoPE tables, compute the
  output using pre-norm architecture with Grouped Query Attention (GQA), Rotary Position Embeddings
  (RoPE), and a SwiGLU feed-forward network.
</p>

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 340 660" width="340" height="660" style="display:block; margin:20px auto;">
  <defs>
    <marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M0 0L10 5L0 10z" fill="#999"/>
    </marker>
  </defs>
  <rect width="340" height="660" fill="#222"/>

  <!-- Input label -->
  <text x="140" y="20" text-anchor="middle" fill="#ccc" font-size="13" font-family="monospace">x (seq_len, 512)</text>

  <!-- Arrow: input -> RMSNorm1 -->
  <line x1="140" y1="28" x2="140" y2="46" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- Residual 1: fork right, down, back left to Add1 -->
  <line x1="140" y1="36" x2="280" y2="36" stroke="#555" stroke-width="1.5" stroke-dasharray="5,4"/>
  <line x1="280" y1="36" x2="280" y2="306" stroke="#555" stroke-width="1.5" stroke-dasharray="5,4"/>
  <line x1="280" y1="306" x2="157" y2="306" stroke="#555" stroke-width="1.5" stroke-dasharray="5,4" marker-end="url(#ah)"/>
  <text x="290" y="175" fill="#666" font-size="10" font-family="sans-serif" transform="rotate(90,290,175)">residual</text>

  <!-- RMSNorm1 -->
  <rect x="60" y="49" width="160" height="30" rx="5" fill="#333" stroke="#777" stroke-width="1"/>
  <text x="140" y="69" text-anchor="middle" fill="#ccc" font-size="12" font-family="sans-serif">RMSNorm 1</text>
  <line x1="140" y1="79" x2="140" y2="97" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- QKV Proj -->
  <rect x="60" y="100" width="160" height="30" rx="5" fill="#1e2d4d" stroke="#4477bb" stroke-width="1"/>
  <text x="140" y="120" text-anchor="middle" fill="#aaccee" font-size="12" font-family="sans-serif">QKV Projection (GQA)</text>
  <line x1="140" y1="130" x2="140" y2="148" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- RoPE -->
  <rect x="60" y="151" width="160" height="30" rx="5" fill="#2d1e4d" stroke="#7755bb" stroke-width="1"/>
  <text x="140" y="171" text-anchor="middle" fill="#ccaaee" font-size="12" font-family="sans-serif">RoPE (Q and K)</text>
  <line x1="140" y1="181" x2="140" y2="199" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- Causal Attn -->
  <rect x="60" y="202" width="160" height="30" rx="5" fill="#1e2d4d" stroke="#4477bb" stroke-width="1"/>
  <text x="140" y="222" text-anchor="middle" fill="#aaccee" font-size="12" font-family="sans-serif">Causal Attention</text>
  <line x1="140" y1="232" x2="140" y2="250" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- Output Proj -->
  <rect x="60" y="253" width="160" height="30" rx="5" fill="#1e2d4d" stroke="#4477bb" stroke-width="1"/>
  <text x="140" y="273" text-anchor="middle" fill="#aaccee" font-size="12" font-family="sans-serif">Output Projection</text>
  <line x1="140" y1="283" x2="140" y2="294" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- Add 1 -->
  <circle cx="140" cy="306" r="12" fill="#222" stroke="#999" stroke-width="1.5"/>
  <text x="140" y="311" text-anchor="middle" fill="#ccc" font-size="15" font-family="sans-serif" font-weight="bold">+</text>
  <line x1="140" y1="318" x2="140" y2="342" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- Residual 2 -->
  <line x1="140" y1="330" x2="280" y2="330" stroke="#555" stroke-width="1.5" stroke-dasharray="5,4"/>
  <line x1="280" y1="330" x2="280" y2="586" stroke="#555" stroke-width="1.5" stroke-dasharray="5,4"/>
  <line x1="280" y1="586" x2="157" y2="586" stroke="#555" stroke-width="1.5" stroke-dasharray="5,4" marker-end="url(#ah)"/>
  <text x="290" y="460" fill="#666" font-size="10" font-family="sans-serif" transform="rotate(90,290,460)">residual</text>

  <!-- RMSNorm2 -->
  <rect x="60" y="345" width="160" height="30" rx="5" fill="#333" stroke="#777" stroke-width="1"/>
  <text x="140" y="365" text-anchor="middle" fill="#ccc" font-size="12" font-family="sans-serif">RMSNorm 2</text>
  <line x1="140" y1="375" x2="140" y2="393" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- Gate + Up Proj -->
  <rect x="60" y="396" width="160" height="30" rx="5" fill="#1e3d2d" stroke="#44aa66" stroke-width="1"/>
  <text x="140" y="416" text-anchor="middle" fill="#aaeebb" font-size="12" font-family="sans-serif">Gate &amp; Up Proj (512→1408)</text>
  <line x1="140" y1="426" x2="140" y2="444" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- SiLU + multiply -->
  <rect x="60" y="447" width="160" height="30" rx="5" fill="#1e3d2d" stroke="#44aa66" stroke-width="1"/>
  <text x="140" y="467" text-anchor="middle" fill="#aaeebb" font-size="12" font-family="sans-serif">SiLU(gate) &#x2299; up</text>
  <line x1="140" y1="477" x2="140" y2="495" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- Down Proj -->
  <rect x="60" y="498" width="160" height="30" rx="5" fill="#1e3d2d" stroke="#44aa66" stroke-width="1"/>
  <text x="140" y="518" text-anchor="middle" fill="#aaeebb" font-size="12" font-family="sans-serif">Down Proj (1408→512)</text>
  <line x1="140" y1="528" x2="140" y2="574" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- Add 2 -->
  <circle cx="140" cy="586" r="12" fill="#222" stroke="#999" stroke-width="1.5"/>
  <text x="140" y="591" text-anchor="middle" fill="#ccc" font-size="15" font-family="sans-serif" font-weight="bold">+</text>
  <line x1="140" y1="598" x2="140" y2="622" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- Output label -->
  <text x="140" y="640" text-anchor="middle" fill="#ccc" font-size="13" font-family="monospace">output (seq_len, 512)</text>
</svg>

<p>
  The block follows Llama's <strong>pre-norm</strong> architecture. Unlike GPT-2, it uses
  <strong>RMSNorm</strong> (no mean subtraction, no additive bias), <strong>Grouped Query
  Attention</strong> with 8 query heads and 2 key/value heads, <strong>Rotary Position
  Embeddings</strong> applied to Q and K, and a <strong>SwiGLU</strong> feed-forward network.
  None of the linear projections have bias terms.
</p>

$$
\begin{aligned}
x' &= x + \text{Attn}\!\left(\text{RMSNorm}_1(x),\; \cos,\; \sin\right) \$$4pt]
\text{output} &= x' + \text{FFN}\!\left(\text{RMSNorm}_2(x')\right)
\end{aligned}
$$

<p>The sub-operations in detail:</p>

$$
\begin{aligned}
\text{RMSNorm}(z, w) &= \frac{z}{\sqrt{\frac{1}{d}\sum_i z_i^2 + \varepsilon}} \odot w, \quad \varepsilon = 10^{-5} \$$8pt]
Q &= \text{RMSNorm}_1(x)\, W_Q^\top \in \mathbb{R}^{T \times 512}, \quad \text{reshape to } (T, 8, 64) \$$4pt]
K &= \text{RMSNorm}_1(x)\, W_K^\top \in \mathbb{R}^{T \times 128}, \quad \text{reshape to } (T, 2, 64) \$$4pt]
V &= \text{RMSNorm}_1(x)\, W_V^\top \in \mathbb{R}^{T \times 128}, \quad \text{reshape to } (T, 2, 64) \$$8pt]
\text{RoPE}(q, \cos, \sin) &: \quad [q_1 \mid q_2] \mapsto [q_1 \odot \cos - q_2 \odot \sin \mid q_1 \odot \sin + q_2 \odot \cos] \$$4pt]
&\quad q_1 = q[\ldots, {:}32],\; q_2 = q[\ldots, {32:}] \$$8pt]
\text{GQA} &: \text{repeat } K,V \text{ along head dim } 4\times \text{ to match 8 Q heads} \$$4pt]
\text{head}_i &= \text{softmax}\!\left(\frac{Q_i K_i^\top}{\sqrt{64}} + M_{\text{causal}}\right) V_i \$$8pt]
\text{Attn}(x) &= \text{Concat}(\text{head}_1, \ldots, \text{head}_8)\; W_O^\top \$$8pt]
\text{FFN}(z) &= \bigl(\text{SiLU}(z\, W_{\text{gate}}^\top) \odot z\, W_{\text{up}}^\top\bigr)\; W_{\text{down}}^\top
\end{aligned}
$$

<p>where $M_{\text{causal}}$ is the upper-triangular causal mask ($-\infty$ above the diagonal)
and $\text{SiLU}(x) = x \cdot \sigma(x)$.</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>output</code> tensor</li>
  <li>RMSNorm uses $\varepsilon = 10^{-5}$, no additive bias</li>
  <li>Apply causal masking: position $i$ attends only to positions $\le i$</li>
  <li>Repeat K and V heads $4\times$ (GQA groups) before computing attention</li>
  <li><code>cos</code> and <code>sin</code> have shape <code>(seq_len, 32)</code> — apply
      them to both Q and K heads independently</li>
</ul>

<h2>Weight Layout</h2>
<p>All parameters are packed into a single contiguous <code>weights</code> buffer
(2,819,072 floats) in the order below. All 2-D matrices are stored row-major
with shape <code>(out_dim, in_dim)</code>. There are no bias terms.</p>

<table style="border-collapse:separate; border-spacing:16px 6px;">
  <tr>
    <th style="text-align:left;">Parameter</th>
    <th style="text-align:left;">Shape</th>
    <th style="text-align:right;">Size</th>
    <th style="text-align:right;">Offset</th>
  </tr>
  <tr>
    <td>$w_1$ (RMSNorm 1 scale)</td>
    <td>(512,)</td>
    <td style="text-align:right;">512</td>
    <td style="text-align:right;">0</td>
  </tr>
  <tr>
    <td>$W_Q$</td>
    <td>(512, 512)</td>
    <td style="text-align:right;">262,144</td>
    <td style="text-align:right;">512</td>
  </tr>
  <tr>
    <td>$W_K$</td>
    <td>(128, 512)</td>
    <td style="text-align:right;">65,536</td>
    <td style="text-align:right;">262,656</td>
  </tr>
  <tr>
    <td>$W_V$</td>
    <td>(128, 512)</td>
    <td style="text-align:right;">65,536</td>
    <td style="text-align:right;">328,192</td>
  </tr>
  <tr>
    <td>$W_O$</td>
    <td>(512, 512)</td>
    <td style="text-align:right;">262,144</td>
    <td style="text-align:right;">393,728</td>
  </tr>
  <tr>
    <td>$w_2$ (RMSNorm 2 scale)</td>
    <td>(512,)</td>
    <td style="text-align:right;">512</td>
    <td style="text-align:right;">655,872</td>
  </tr>
  <tr>
    <td>$W_{\text{gate}}$</td>
    <td>(1408, 512)</td>
    <td style="text-align:right;">720,896</td>
    <td style="text-align:right;">656,384</td>
  </tr>
  <tr>
    <td>$W_{\text{up}}$</td>
    <td>(1408, 512)</td>
    <td style="text-align:right;">720,896</td>
    <td style="text-align:right;">1,377,280</td>
  </tr>
  <tr>
    <td>$W_{\text{down}}$</td>
    <td>(512, 1408)</td>
    <td style="text-align:right;">720,896</td>
    <td style="text-align:right;">2,098,176</td>
  </tr>
</table>

<h2>Example</h2>
<p>With <code>seq_len</code> = 4, <code>x</code> drawn uniformly from [&minus;1, 1], and randomly
initialized weights:</p>
<pre>
Input:  x.shape       = (4, 512)       # 4 token hidden states
        weights.shape = (2,819,072,)   # packed weight buffer
        cos.shape     = (4, 32)        # precomputed RoPE cosines
        sin.shape     = (4, 32)        # precomputed RoPE sines
        seq_len       = 4
Output: output.shape  = (4, 512)       # transformed token hidden states
</pre>

<h2>Constraints</h2>
<ul>
  <li><code>d_model</code> = 512, <code>n_q_heads</code> = 8, <code>n_kv_heads</code> = 2,
      <code>head_dim</code> = 64, <code>ffn_hidden</code> = 1,408</li>
  <li>1 &le; <code>seq_len</code> &le; 4,096</li>
  <li>All tensors use 32-bit floating point</li>
  <li>Performance is measured with <code>seq_len</code> = 2,048</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// x, output, weights, cos, sin are device pointers
extern "C" void solve(const float* x, float* output, const float* weights, const float* cos,
                      const float* sin, int seq_len) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# x, output, weights, cos, sin are tensors on the GPU
@cute.jit
def solve(
    x: cute.Tensor,
    output: cute.Tensor,
    weights: cute.Tensor,
    cos: cute.Tensor,
    sin: cute.Tensor,
    seq_len: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# x, weights, cos, sin are tensors on GPU
@jax.jit
def solve(
    x: jax.Array,
    weights: jax.Array,
    cos: jax.Array,
    sin: jax.Array,
    seq_len: int,
) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# x, output, weights, cos, sin are device pointers
@export
def solve(
    x: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    weights: UnsafePointer[Float32, MutExternalOrigin],
    cos: UnsafePointer[Float32, MutExternalOrigin],
    sin: UnsafePointer[Float32, MutExternalOrigin],
    seq_len: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# x, output, weights, cos, sin are tensors on the GPU
def solve(
    x: torch.Tensor,
    output: torch.Tensor,
    weights: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
    seq_len: int,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# x, output, weights, cos, sin are tensors on the GPU
def solve(
    x: torch.Tensor,
    output: torch.Tensor,
    weights: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
    seq_len: int,
):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
import math
from typing import Any, Dict, List

import torch
import torch.nn.functional as F

# Llama-style architecture constants
D = 512  # model dimension
NUM_Q_HEADS = 8  # number of query heads
NUM_KV_HEADS = 2  # number of key/value heads (grouped query attention)
HEAD_DIM = D // NUM_Q_HEADS  # = 64
Q_DIM = NUM_Q_HEADS * HEAD_DIM  # = 512
KV_DIM = NUM_KV_HEADS * HEAD_DIM  # = 128
GQA_GROUPS = NUM_Q_HEADS // NUM_KV_HEADS  # = 4
FFN_HIDDEN = 1408  # SwiGLU intermediate dimension

# Weight buffer layout offsets (all projections stored as (out_dim, in_dim))
O_RMS1_W = 0
O_WQ = O_RMS1_W + D  # Q projection: Q_DIM x D
O_WK = O_WQ + Q_DIM * D  # K projection: KV_DIM x D
O_WV = O_WK + KV_DIM * D  # V projection: KV_DIM x D
O_WO = O_WV + KV_DIM * D  # output projection: D x D
O_RMS2_W = O_WO + D * D  # RMS norm 2 weights: D
O_WGATE = O_RMS2_W + D  # gate projection: FFN_HIDDEN x D
O_WUP = O_WGATE + FFN_HIDDEN * D  # up projection: FFN_HIDDEN x D
O_WDOWN = O_WUP + FFN_HIDDEN * D  # down projection: D x FFN_HIDDEN
TOTAL_WEIGHTS = O_WDOWN + D * FFN_HIDDEN


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Llama Transformer Block",
            atol=1e-03,
            rtol=1e-03,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(
        self,
        x: torch.Tensor,
        output: torch.Tensor,
        weights: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor,
        seq_len: int,
    ):
        assert x.shape == (seq_len, D)
        assert output.shape == (seq_len, D)
        assert weights.shape == (TOTAL_WEIGHTS,)
        assert cos.shape == (seq_len, HEAD_DIM // 2)
        assert sin.shape == (seq_len, HEAD_DIM // 2)
        assert x.dtype == output.dtype == weights.dtype == cos.dtype == sin.dtype
        assert x.device.type == "cuda"
        assert output.device.type == "cuda"
        assert weights.device.type == "cuda"
        assert cos.device.type == "cuda"
        assert sin.device.type == "cuda"

        def rms_norm(z, w):
            return z * torch.rsqrt(z.pow(2).mean(-1, keepdim=True) + 1e-5) * w

        def apply_rope(qk, c, s):
            # qk: (seq_len, num_heads, head_dim)
            q1, q2 = qk[..., : HEAD_DIM // 2], qk[..., HEAD_DIM // 2 :]
            c = c.unsqueeze(1)  # (seq_len, 1, head_dim//2)
            s = s.unsqueeze(1)
            return torch.cat([q1 * c - q2 * s, q1 * s + q2 * c], dim=-1)

        # unpack weights
        rms1_w = weights[O_RMS1_W:O_WQ]
        W_Q = weights[O_WQ:O_WK].view(Q_DIM, D)
        W_K = weights[O_WK:O_WV].view(KV_DIM, D)
        W_V = weights[O_WV:O_WO].view(KV_DIM, D)
        W_O = weights[O_WO:O_RMS2_W].view(D, D)
        rms2_w = weights[O_RMS2_W:O_WGATE]
        W_gate = weights[O_WGATE:O_WUP].view(FFN_HIDDEN, D)
        W_up = weights[O_WUP:O_WDOWN].view(FFN_HIDDEN, D)
        W_down = weights[O_WDOWN:TOTAL_WEIGHTS].view(D, FFN_HIDDEN)

        # --- Attention sub-block ---
        x_norm = rms_norm(x, rms1_w)

        # QKV projections
        q = (x_norm @ W_Q.T).view(seq_len, NUM_Q_HEADS, HEAD_DIM)
        k = (x_norm @ W_K.T).view(seq_len, NUM_KV_HEADS, HEAD_DIM)
        v = (x_norm @ W_V.T).view(seq_len, NUM_KV_HEADS, HEAD_DIM)

        # Apply RoPE to Q and K
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # Reshape for batched matmul: (num_heads, seq_len, head_dim)
        q = q.transpose(0, 1)  # (NUM_Q_HEADS, seq_len, HEAD_DIM)
        k = k.transpose(0, 1)  # (NUM_KV_HEADS, seq_len, HEAD_DIM)
        v = v.transpose(0, 1)  # (NUM_KV_HEADS, seq_len, HEAD_DIM)

        # GQA: broadcast K and V to match Q heads
        k = k.repeat_interleave(GQA_GROUPS, dim=0)  # (NUM_Q_HEADS, seq_len, HEAD_DIM)
        v = v.repeat_interleave(GQA_GROUPS, dim=0)

        # Causal scaled dot-product attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(HEAD_DIM)
        causal_mask = torch.triu(
            torch.full((seq_len, seq_len), float("-inf"), device=x.device, dtype=x.dtype),
            diagonal=1,
        )
        scores = scores + causal_mask
        attn_weights = torch.softmax(scores, dim=-1)
        attn_out = torch.matmul(attn_weights, v)  # (NUM_Q_HEADS, seq_len, HEAD_DIM)

        # Merge heads and project
        attn_out = attn_out.transpose(0, 1).contiguous().view(seq_len, D)
        attn_proj = attn_out @ W_O.T

        # Residual 1
        hidden = x + attn_proj

        # --- FFN sub-block ---
        h_norm = rms_norm(hidden, rms2_w)

        # SwiGLU: gate * up, then project down
        gate = F.silu(h_norm @ W_gate.T)
        up = h_norm @ W_up.T
        ffn_out = (gate * up) @ W_down.T

        # Residual 2
        output.copy_(hidden + ffn_out)

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "x": (ctypes.POINTER(ctypes.c_float), "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "weights": (ctypes.POINTER(ctypes.c_float), "in"),
            "cos": (ctypes.POINTER(ctypes.c_float), "in"),
            "sin": (ctypes.POINTER(ctypes.c_float), "in"),
            "seq_len": (ctypes.c_int, "in"),
        }

    def _make_rope_tables(self, seq_len, device, dtype, theta=10000.0):
        positions = torch.arange(seq_len, device=device, dtype=dtype)
        freqs = 1.0 / (
            theta ** (torch.arange(0, HEAD_DIM, 2, device=device, dtype=dtype) / HEAD_DIM)
        )
        angles = torch.outer(positions, freqs)  # (seq_len, HEAD_DIM//2)
        return angles.cos(), angles.sin()

    def _make_weights(self, device, dtype):
        scale = 0.02
        rms1_w = torch.empty(D, device=device, dtype=dtype).uniform_(0.8, 1.2)
        W_Q = torch.empty(Q_DIM, D, device=device, dtype=dtype).normal_(0, scale)
        W_K = torch.empty(KV_DIM, D, device=device, dtype=dtype).normal_(0, scale)
        W_V = torch.empty(KV_DIM, D, device=device, dtype=dtype).normal_(0, scale)
        W_O = torch.empty(D, D, device=device, dtype=dtype).normal_(0, scale)
        rms2_w = torch.empty(D, device=device, dtype=dtype).uniform_(0.8, 1.2)
        W_gate = torch.empty(FFN_HIDDEN, D, device=device, dtype=dtype).normal_(0, scale)
        W_up = torch.empty(FFN_HIDDEN, D, device=device, dtype=dtype).normal_(0, scale)
        W_down = torch.empty(D, FFN_HIDDEN, device=device, dtype=dtype).normal_(0, scale)
        return torch.cat(
            [
                rms1_w,
                W_Q.flatten(),
                W_K.flatten(),
                W_V.flatten(),
                W_O.flatten(),
                rms2_w,
                W_gate.flatten(),
                W_up.flatten(),
                W_down.flatten(),
            ]
        )

    def _make_test_case(self, seq_len, zero_x=False):
        dtype = torch.float32
        device = "cuda"
        weights = self._make_weights(device, dtype)
        cos, sin = self._make_rope_tables(seq_len, device, dtype)
        if zero_x:
            x = torch.zeros(seq_len, D, device=device, dtype=dtype)
        else:
            x = torch.empty(seq_len, D, device=device, dtype=dtype).uniform_(-1.0, 1.0)
        return {
            "x": x,
            "output": torch.empty(seq_len, D, device=device, dtype=dtype),
            "weights": weights,
            "cos": cos,
            "sin": sin,
            "seq_len": seq_len,
        }

    def generate_example_test(self) -> Dict[str, Any]:
        torch.manual_seed(0)
        return self._make_test_case(4)

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        tests = []
        # single token (decode phase)
        tests.append(self._make_test_case(1))
        # zero input
        tests.append(self._make_test_case(4, zero_x=True))
        # small edge cases
        tests.append(self._make_test_case(2))
        tests.append(self._make_test_case(4))
        # power-of-2
        tests.append(self._make_test_case(16))
        tests.append(self._make_test_case(64))
        # non-power-of-2
        tests.append(self._make_test_case(30))
        tests.append(self._make_test_case(100))
        # realistic inference lengths
        tests.append(self._make_test_case(128))
        tests.append(self._make_test_case(256))
        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        return self._make_test_case(2048)


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
